In [0]:
#Find most visited floor and total visits for each user

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define schema
schema = StructType([
    StructField("name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("email", StringType(), True),
    StructField("floor", IntegerType(), True),
    StructField("resources", StringType(), True)
])

# Data to insert
data = [
    ('A','Bangalore','A@gmail.com',1,'CPU'),
    ('A','Bangalore','A1@gmail.com',1,'CPU'),
    ('A','Bangalore','A2@gmail.com',2,'DESKTOP'),
    ('B','Bangalore','B@gmail.com',2,'DESKTOP'),
    ('B','Bangalore','B1@gmail.com',2,'DESKTOP'),
    ('B','Bangalore','B2@gmail.com',1,'MONITOR')
]

# Create DataFrame
entries_df = spark.createDataFrame(data, schema)

display(entries_df)

In [0]:
entries_df.createOrReplaceTempView("entries")

In [0]:
spark.sql("""
        with cte1 as (
            select 
                name, 
                floor, 
                rank() over(partition by name order by count(1) desc) as rn 
            from entries
            group by name, floor
        ),
        cte2 as (
            select 
                name, 
                count(1) as total_visits,
                string_agg(resources, ",") as resources_used
            from entries 
            group by name
        )
        select 
            c1.name, 
            c1.floor as most_visited_floor, 
            c2.total_visits,
            c2.resources_used
        from cte1 c1 
        join cte2 c2 on c1.name = c2.name 
        where c1.rn = 1; 
""").display()

In [0]:
spark.sql("""
        with cte1 as (
            select 
                name, 
                floor, 
                rank() over(partition by name order by count(1) desc) as rn 
            from entries
            group by name, floor
        ),
        cte2 as (
            select 
                name, 
                count(1) as total_visits,
                string_agg(resources, ",") as resources_used
            from entries 
            group by name
        )
        select 
            c1.name, 
            c1.floor as most_visited_floor, 
            c2.total_visits,
            c2.resources_used
        from cte1 c1 
        join cte2 c2 on c1.name = c2.name 
        where c1.rn = 1; 
""").display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# cte1: rank floors by visit count per name
floor_count_df = entries_df.groupBy("name", "floor").count()


In [0]:

window_spec = Window.partitionBy("name").orderBy(F.col("count").desc())
cte1 = floor_count_df.withColumn("rn", F.rank().over(window_spec))

# cte2: total visits and resources used per name
cte2 = entries_df.groupBy("name").agg(
    F.count("*").alias("total_visits"),
    F.concat_ws(",", F.collect_list("resources")).alias("resources_used")
)

# join cte1 and cte2, filter for most visited floor
result_df = cte1.filter(F.col("rn") == 1).join(cte2, "name") \
    .select(
        F.col("name"),
        F.col("floor").alias("most_visited_floor"),
        F.col("total_visits"),
        F.col("resources_used")
    )

display(result_df)